<a href="https://colab.research.google.com/github/subbareddypullalacheruvu/Major-Project-Robust-Fraud-Detection-with-Spatial-Temporal-Features-and-Class-Imbalance-Learning-/blob/main/notebooks/final_model_comparison.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Final model comparison

**Weeks 8–9.** Before moving on to IEEE-CIS validation, we wanted to know whether a different model could beat our best result so far. This notebook:

1. adds two new models, **Balanced Random Forest** and **CatBoost**, on the same six-feature Sparkov set;
2. re-scores the strongest earlier models (**Random Forest** and **XGBoost Weighted**) under the same protocol, so every model is compared on equal terms;
3. tunes the two strongest candidates using the **validation set only**;
4. selects the single model we carry into Week 9 (IEEE-CIS validation, ablation and SHAP).

**Evaluation protocol (identical for every model):**
- Features: `amt_log`, `category_risk`, `is_night`, `velocity_log`, `age`, `gender_flag`.
- Training data: 80% of `sparkov_train`; validation: the remaining 20% (stratified).
- Test: Kaggle's `fraudTest.csv`, a separate and later time window (from `split_sparkov()`).
- Threshold: chosen per model by maximising F2 on validation.
- Test is used once per model, for the final reported scores only.
- Metrics: PR-AUC, F1, F2 and MCC.

## 1. Environment setup

Download ULB and Sparkov from Kaggle, clone the project repository (which contains `src/split.py` and `src/utils.py`), and move the data into the repository's `data/` folder. Credentials are typed in at runtime with `getpass` and are never stored in the notebook.

In [1]:
# 1.1 Download both datasets from Kaggle
import os
from getpass import getpass

os.environ['KAGGLE_USERNAME'] = input('Kaggle username: ')
os.environ['KAGGLE_KEY'] = getpass('Kaggle API key: ')

!kaggle datasets download -d mlg-ulb/creditcardfraud -p /content/data/ulb --unzip
!kaggle datasets download -d kartik2112/fraud-detection -p /content/data/sparkov --unzip

Kaggle username: Subbareddy_subbz
Kaggle API key: ··········
Dataset URL: https://www.kaggle.com/datasets/mlg-ulb/creditcardfraud
License(s): DbCL-1.0
100% 66.0M/66.0M [00:01<00:00, 64.9MB/s]

Dataset URL: https://www.kaggle.com/datasets/kartik2112/fraud-detection
License(s): CC0-1.0
100% 202M/202M [00:03<00:00, 62.8MB/s]



In [2]:
# 1.2 Clone the project repository (contains src/split.py and src/utils.py)
github_token = getpass('Paste your GitHub token: ')
repo_url = f'https://{github_token}@github.com/subbareddypullalacheruvu/Major-Project-Robust-Fraud-Detection-with-Spatial-Temporal-Features-and-Class-Imbalance-Learning-.git'
!git clone {repo_url}

Paste your GitHub token: ··········
Cloning into 'Major-Project-Robust-Fraud-Detection-with-Spatial-Temporal-Features-and-Class-Imbalance-Learning-'...
remote: Enumerating objects: 151, done.
remote: Counting objects: 100% (151/151), done.
remote: Compressing objects: 100% (143/143), done.
remote: Total 151 (delta 95), reused 18 (delta 5), pack-reused 0 (from 0)
Receiving objects: 100% (151/151), 423.79 KiB | 8.31 MiB/s, done.
Resolving deltas: 100% (95/95), done.


In [3]:
# 1.3 Move into the repository and place the downloaded data in data/
%cd Major-Project-Robust-Fraud-Detection-with-Spatial-Temporal-Features-and-Class-Imbalance-Learning-
!mv /content/data/ulb /content/data/sparkov data/

/content/Major-Project-Robust-Fraud-Detection-with-Spatial-Temporal-Features-and-Class-Imbalance-Learning-


In [4]:
# 1.4 Install the two libraries that are not preinstalled on Colab
!pip install -q catboost imbalanced-learn

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.1/97.1 MB 8.4 MB/s eta 0:00:00


In [5]:
# 1.5 Imports and project helper functions
import sys
import numpy as np
import pandas as pd

sys.path.append('/content/Major-Project-Robust-Fraud-Detection-with-Spatial-Temporal-Features-and-Class-Imbalance-Learning-')
from src.split import split_sparkov
from src.utils import fit_category_risk, apply_category_risk, add_prev_txn_features

## 2. Six-feature Sparkov set

We rebuild exactly the same six features used in `feature_engineering.ipynb` and `boosting_comparison.ipynb`, so the results here are directly comparable with earlier stages.

Two details matter for leakage:
- `category_risk` is **fit on the training split only** and then applied to both splits.
- Implied velocity is computed on the combined, time-sorted train + test timeline, so the first test transaction of each card can still look back at that card's last training transaction. It uses only earlier timestamps and merchant locations, never fraud labels.

Because the train and test DataFrames have overlapping index labels, we tag each row with a `_split` marker and use that to separate them again, rather than relying on the index.

In [6]:
# 2.1 Load Kaggle's chronological train/test split
sparkov_train, sparkov_test = split_sparkov()

# category risk: fit on train only, then apply to both splits
category_risk_map = fit_category_risk(sparkov_train, 'category', 'is_fraud')
sparkov_train['category_risk'] = apply_category_risk(sparkov_train, 'category', category_risk_map)
sparkov_test['category_risk']  = apply_category_risk(sparkov_test, 'category', category_risk_map)

# mark which split each row came from, since train/test indexes overlap
sparkov_train['_split'] = 'train'
sparkov_test['_split']  = 'test'

# implied velocity on the full timeline, so boundary transactions
# still have a real previous transaction to compare against
full = pd.concat([sparkov_train, sparkov_test]).sort_values('trans_date_trans_time')
full = add_prev_txn_features(full, 'cc_num', 'trans_date_trans_time', 'merch_lat', 'merch_long')

# split back using the marker column, not the index
sparkov_train = full[full['_split'] == 'train'].drop(columns='_split')
sparkov_test  = full[full['_split'] == 'test'].drop(columns='_split')

print(sparkov_train.shape, sparkov_test.shape)

Train: 1,296,675 rows, from 2019-01-01 00:00:18 to 2020-06-21 12:13:37
Test:  555,719 rows, from 2020-06-21 12:14:25 to 2020-12-31 23:59:34
(1296675, 27) (555719, 27)


In [7]:
# 2.2 Build the six model features

# velocity_log: a card's first-ever transaction has no previous one -> 0;
# log1p tames the extreme outliers (log beat clipping in feature engineering)
sparkov_train['implied_velocity_kmh'] = sparkov_train['implied_velocity_kmh'].fillna(0)
sparkov_test['implied_velocity_kmh']  = sparkov_test['implied_velocity_kmh'].fillna(0)
sparkov_train['velocity_log'] = np.log1p(sparkov_train['implied_velocity_kmh'])
sparkov_test['velocity_log']  = np.log1p(sparkov_test['implied_velocity_kmh'])

# is_night: 10pm-3am, the high-fraud window found in the EDA
sparkov_train['hour'] = pd.to_datetime(sparkov_train['trans_date_trans_time']).dt.hour
sparkov_test['hour']  = pd.to_datetime(sparkov_test['trans_date_trans_time']).dt.hour
sparkov_train['is_night'] = sparkov_train['hour'].isin([22, 23, 0, 1, 2, 3]).astype(int)
sparkov_test['is_night']  = sparkov_test['hour'].isin([22, 23, 0, 1, 2, 3]).astype(int)

# amt_log: raw amount is heavily right-skewed
sparkov_train['amt_log'] = np.log1p(sparkov_train['amt'])
sparkov_test['amt_log']  = np.log1p(sparkov_test['amt'])

# age in years at the time of the transaction
sparkov_train['age'] = (pd.to_datetime(sparkov_train['trans_date_trans_time']) - pd.to_datetime(sparkov_train['dob'])).dt.days / 365.25
sparkov_test['age']  = (pd.to_datetime(sparkov_test['trans_date_trans_time'])  - pd.to_datetime(sparkov_test['dob'])).dt.days / 365.25

# gender_flag: 1 = male, 0 = female
sparkov_train['gender_flag'] = (sparkov_train['gender'] == 'M').astype(int)
sparkov_test['gender_flag']  = (sparkov_test['gender'] == 'M').astype(int)

# final six-feature matrices
features = ['amt_log', 'category_risk', 'is_night', 'velocity_log', 'age', 'gender_flag']
target = 'is_fraud'

X_train, y_train = sparkov_train[features], sparkov_train[target]
X_test, y_test = sparkov_test[features], sparkov_test[target]

print(X_train.shape, X_test.shape)
print(y_train.mean(), y_test.mean())

(1296675, 6) (555719, 6)
0.005788651743883394 0.0038598644278853163


Row counts and fraud rates (train ≈0.579%, test ≈0.386%) match every earlier stage of the project.

### 2.3 Train/validation split

We carve 20% of the training data off as a validation set. It is used for threshold tuning and hyperparameter selection, so the test set stays untouched until the final evaluation. Stratifying keeps the rare fraud rate the same in both pieces, and `random_state=42` reproduces the exact split used in `boosting_comparison.ipynb`.

In [8]:
from sklearn.model_selection import train_test_split

# stratified 80/20 split of the training data; test is not involved
X_train_sub, X_val, y_train_sub, y_val = train_test_split(
    X_train, y_train,
    test_size=0.2,
    stratify=y_train,
    random_state=42
)

print(X_train_sub.shape, X_val.shape, X_test.shape)
print(y_train_sub.mean(), y_val.mean(), y_test.mean())

(1037340, 6) (259335, 6) (555719, 6)
0.00578884454470087 0.005787880540613492 0.0038598644278853163


In [9]:
# shorter names for the model-training data used in the rest of the notebook
X_tr, y_tr = X_train_sub, y_train_sub

# sanity check: sizes, fraud rates and feature list for all three pieces
for name, X, y in [("train", X_tr, y_tr), ("val", X_val, y_val), ("test", X_test, y_test)]:
    print(f"{name:5s}  rows={len(X):>9,}  fraud rate={y.mean():.4%}")
print("features:", list(X_tr.columns))

train  rows=1,037,340  fraud rate=0.5789%
val    rows=  259,335  fraud rate=0.5788%
test   rows=  555,719  fraud rate=0.3860%
features: ['amt_log', 'category_risk', 'is_night', 'velocity_log', 'age', 'gender_flag']


The data is identical to the boosting notebook, so every model in this notebook is trained and evaluated on exactly the same rows.

## 3. Shared evaluation function

Every model goes through the same function, which applies the protocol in two steps:
1. **Validation:** sweep thresholds from 0.01 to 0.99 and keep the one with the highest F2.
2. **Test:** apply that threshold once and report PR-AUC, F1, F2 and MCC.

Validation PR-AUC is also reported, so we can see the gap between validation and test performance.

In [10]:
from sklearn.metrics import average_precision_score, f1_score, fbeta_score, matthews_corrcoef

def tune_and_evaluate(model, name):
    """Pick the F2-optimal threshold on validation, then score once on test."""
    # step 1: threshold chosen on validation only
    val_proba = model.predict_proba(X_val)[:, 1]
    thresholds = np.arange(0.01, 1.00, 0.01)
    f2_scores = [fbeta_score(y_val, (val_proba >= t).astype(int), beta=2) for t in thresholds]
    best_t = thresholds[int(np.argmax(f2_scores))]

    # step 2: a single evaluation on the held-out test set
    test_proba = model.predict_proba(X_test)[:, 1]
    test_pred = (test_proba >= best_t).astype(int)
    result = {
        "Model": name,
        "Threshold": round(best_t, 2),
        "Val PR-AUC": round(average_precision_score(y_val, val_proba), 4),
        "PR-AUC": round(average_precision_score(y_test, test_proba), 4),
        "F1": round(f1_score(y_test, test_pred), 4),
        "F2": round(fbeta_score(y_test, test_pred, beta=2), 4),
        "MCC": round(matthews_corrcoef(y_test, test_pred), 4),
    }
    print(result)
    return result

# collects one result row per model
results = []

## 4. New models

### 4.1 Balanced Random Forest

Balanced Random Forest (`imbalanced-learn`) builds imbalance handling into the model itself: each tree is trained on all fraud cases plus an equal-sized random sample of legitimate transactions. It is a natural candidate because plain Random Forest was our strongest model so far.

In [11]:
from imblearn.ensemble import BalancedRandomForestClassifier

brf = BalancedRandomForestClassifier(
    n_estimators=100,
    sampling_strategy="all",   # balance the sample used by every tree
    replacement=True,          # with bootstrap=False: the original Balanced RF formulation
    bootstrap=False,
    random_state=42,
    n_jobs=-1,
)
brf.fit(X_tr, y_tr)
results.append(tune_and_evaluate(brf, "Balanced RF"))

{'Model': 'Balanced RF', 'Threshold': np.float64(0.87), 'Val PR-AUC': np.float64(0.7972), 'PR-AUC': np.float64(0.7207), 'F1': 0.6326, 'F2': 0.7513, 'MCC': np.float64(0.6541)}


The selected threshold is high (0.87). This is expected: every tree sees a 50/50 class mix, so the forest's raw fraud probabilities are inflated and a stricter cut-off is needed. F2 (0.7513) is competitive, but PR-AUC (0.7207) is clearly lower than our earlier tree models. Each tree sees only about 12,000 rows, roughly 1% of the legitimate transactions, so the forest learns less about normal spending behaviour and ranks transactions less well overall.

### 4.2 CatBoost

CatBoost is a gradient-boosting library whose default settings (ordered boosting, symmetric trees) make it less prone to overfitting than XGBoost or LightGBM. We train it twice, mirroring the XGBoost/LightGBM stage: once with no imbalance handling (Control) and once with `scale_pos_weight` (Weighted).

In [12]:
from catboost import CatBoostClassifier

# CatBoost, Control: no imbalance handling
cb_control = CatBoostClassifier(
    iterations=500,
    random_seed=42,
    verbose=0,
    thread_count=-1,
)
cb_control.fit(X_tr, y_tr)
results.append(tune_and_evaluate(cb_control, "CatBoost Control"))

{'Model': 'CatBoost Control', 'Threshold': np.float64(0.12), 'Val PR-AUC': np.float64(0.8744), 'PR-AUC': np.float64(0.8057), 'F1': 0.661, 'F2': 0.7632, 'MCC': np.float64(0.6766)}


In [13]:
# CatBoost, Weighted: same weighting formula as XGBoost Weighted,
# scale_pos_weight = (# legitimate) / (# fraud) in the training data
neg, pos = (y_tr == 0).sum(), (y_tr == 1).sum()
spw = neg / pos
print("scale_pos_weight:", round(spw, 2))

cb_weighted = CatBoostClassifier(
    iterations=500,
    scale_pos_weight=spw,
    random_seed=42,
    verbose=0,
    thread_count=-1,
)
cb_weighted.fit(X_tr, y_tr)
results.append(tune_and_evaluate(cb_weighted, "CatBoost Weighted"))

scale_pos_weight: 171.75
{'Model': 'CatBoost Weighted', 'Threshold': np.float64(0.91), 'Val PR-AUC': np.float64(0.8663), 'PR-AUC': np.float64(0.784), 'F1': 0.6421, 'F2': 0.754, 'MCC': np.float64(0.6611)}


Untreated CatBoost is strong (test PR-AUC 0.8057). Weighting lowered it to 0.7840. This is the opposite of XGBoost, where the same weighting improved PR-AUC, and adds to the LightGBM finding that the effect of reweighting depends on the library rather than being reliably helpful.

## 5. Re-scoring the earlier leaders under the same protocol

Our earlier Random Forest result (test PR-AUC 0.8146, in `imbalance_comparison.ipynb`) was trained on the full training set and evaluated at the default 0.5 threshold, while the boosting models used a validation-tuned threshold. To compare fairly, we retrain Random Forest and XGBoost Weighted here on the same 80% training split, with the same threshold procedure as the new models.

In [14]:
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier

# Random Forest, default settings, no imbalance handling
rf = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
rf.fit(X_tr, y_tr)
results.append(tune_and_evaluate(rf, "Random Forest Control"))

# XGBoost Weighted, identical settings to boosting_comparison.ipynb
xgb_w = XGBClassifier(n_estimators=100, eval_metric="logloss",
                      scale_pos_weight=spw, random_state=42)
xgb_w.fit(X_tr, y_tr)
results.append(tune_and_evaluate(xgb_w, "XGBoost Weighted"))

{'Model': 'Random Forest Control', 'Threshold': np.float64(0.15), 'Val PR-AUC': np.float64(0.8663), 'PR-AUC': np.float64(0.807), 'F1': 0.6957, 'F2': 0.776, 'MCC': np.float64(0.705)}
{'Model': 'XGBoost Weighted', 'Threshold': np.float64(0.93), 'Val PR-AUC': np.float64(0.8716), 'PR-AUC': np.float64(0.7954), 'F1': 0.6668, 'F2': 0.7578, 'MCC': np.float64(0.6791)}


XGBoost Weighted reproduces its earlier test PR-AUC exactly (0.7954), which confirms this notebook's data and protocol match the boosting stage.

### 5.1 Comparison of all models (default settings)

In [15]:
# one row per model, sorted by test PR-AUC
comparison = pd.DataFrame(results).sort_values("PR-AUC", ascending=False).reset_index(drop=True)
comparison

,Model,Threshold,Val PR-AUC,PR-AUC,F1,F2,MCC
0,Random Forest Control,0.15,0.8663,0.8070,0.6957,0.7760,0.7050
1,CatBoost Control,0.12,0.8744,0.8057,0.6610,0.7632,0.6766
2,XGBoost Weighted,0.93,0.8716,0.7954,0.6668,0.7578,0.6791
3,CatBoost Weighted,0.91,0.8663,0.7840,0.6421,0.7540,0.6611
4,Balanced RF,0.87,0.7972,0.7207,0.6326,0.7513,0.6541


**Observations:**
- Random Forest and CatBoost are effectively tied on test PR-AUC (0.8070 vs 0.8057). Random Forest is stronger on F1, F2 and MCC, while CatBoost has the highest validation PR-AUC (0.8744).
- Under the shared protocol, Random Forest's numbers differ from the earlier stage: PR-AUC 0.8146 → 0.8070 (it now trains on 80% of the data), F2 0.7389 → 0.7760 (tuned threshold), MCC 0.7761 → 0.7050 (an F2-optimal threshold trades some precision for recall). The figures in this table supersede the earlier ones, because every model is now evaluated the same way.
- Building imbalance handling into the model (Balanced RF) came last. A plain tree ensemble with a validation-tuned threshold outperformed every treated variant.

Because validation and test do not agree on a single winner at default settings, we do not select a model from this table. Instead, we take the two strongest candidates, Random Forest and CatBoost, tune both, and make the final choice on validation PR-AUC alone.

## 6. Hyperparameter tuning (validation only)

Both models are tuned by comparing **validation PR-AUC only**; the test set is not used in this section.

The tuning below was run once with `RUN_TUNING = True` (about 80 minutes in total on Colab's CPU), and its outputs are kept in the notebook. The switch is set to `False` so that re-running the notebook skips this step, and the selected settings are written directly into Section 7.

In [16]:
# set to True only to repeat the ~80-minute tuning run; the recorded outputs below come from that run
RUN_TUNING = False

### 6.1 Random Forest

With only six features, Random Forest's default `max_features="sqrt"` lets each split consider just 2 features. We test allowing 3 or 4, and requiring a minimum number of samples per leaf (`min_samples_leaf`) to stop trees from fitting very small groups of fraud cases. We also use 200 trees instead of 100.

In [17]:
if RUN_TUNING:
    rf_settings = [
        {"max_features": "sqrt", "min_samples_leaf": 1},   # default settings, for reference
        {"max_features": 3,      "min_samples_leaf": 1},
        {"max_features": 4,      "min_samples_leaf": 1},
        {"max_features": "sqrt", "min_samples_leaf": 5},
        {"max_features": 3,      "min_samples_leaf": 5},
        {"max_features": 3,      "min_samples_leaf": 10},
    ]

    rf_tuning = []
    for s in rf_settings:
        m = RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1, **s)
        m.fit(X_tr, y_tr)
        # each setting is judged on validation PR-AUC only
        val_pr = average_precision_score(y_val, m.predict_proba(X_val)[:, 1])
        rf_tuning.append({**s, "Val PR-AUC": round(val_pr, 4)})
        print(s, "->", round(val_pr, 4))

    display(pd.DataFrame(rf_tuning).sort_values("Val PR-AUC", ascending=False))

- Doubling the trees alone raised validation PR-AUC from 0.8663 to 0.8713.
- Allowing 3 features per split gave the largest single gain (0.8779); 4 features gave no further benefit.
- A minimum of 5 samples per leaf added more (0.8837). Raising it to 10 was slightly worse (0.8805), so 5 is close to the best value in that direction.

**Selected:** `n_estimators=200`, `max_features=3`, `min_samples_leaf=5`.

### 6.2 CatBoost

We vary tree depth and learning rate, and let CatBoost train for up to 2,000 rounds with early stopping: training stops once validation PR-AUC has not improved for 100 rounds.

In [18]:
if RUN_TUNING:
    cb_tuning = []
    for depth in [6, 8]:
        for lr in [0.03, 0.1]:
            m = CatBoostClassifier(
                iterations=2000, depth=depth, learning_rate=lr,
                eval_metric="PRAUC", early_stopping_rounds=100,   # stop when validation PR-AUC stalls
                random_seed=42, verbose=0, thread_count=-1,
            )
            m.fit(X_tr, y_tr, eval_set=(X_val, y_val))
            val_pr = average_precision_score(y_val, m.predict_proba(X_val)[:, 1])
            cb_tuning.append({"depth": depth, "learning_rate": lr,
                              "best_iteration": m.get_best_iteration(),
                              "Val PR-AUC": round(val_pr, 4)})
            print(cb_tuning[-1])

    display(pd.DataFrame(cb_tuning).sort_values("Val PR-AUC", ascending=False))

All four settings fall within 0.002 of each other (0.8802–0.8820), so CatBoost is not sensitive to these settings. Most of its improvement over the default (0.8744) comes from training for more rounds with early stopping.

**Selected:** `depth=8`, `learning_rate=0.1`.

Tuned Random Forest (0.8837) is ahead of tuned CatBoost (0.8820) on validation, even though CatBoost had a small advantage here from using validation for early stopping. **Random Forest is therefore selected on validation alone, before the test set is used.**

## 7. Final evaluation of the tuned models

Both tuned models are retrained with their selected settings and evaluated once on the test set. This is the final test evaluation for this comparison: no settings are changed after seeing these results.

In [19]:
# settings selected in Section 6 (validation PR-AUC only)
rf_tuned = RandomForestClassifier(
    n_estimators=200, max_features=3, min_samples_leaf=5,
    random_state=42, n_jobs=-1,
)
rf_tuned.fit(X_tr, y_tr)

cb_tuned = CatBoostClassifier(
    iterations=2000, depth=8, learning_rate=0.1,
    eval_metric="PRAUC", early_stopping_rounds=100,
    random_seed=42, verbose=0, thread_count=-1,
)
cb_tuned.fit(X_tr, y_tr, eval_set=(X_val, y_val))

# one threshold-tuned evaluation per model; the winner is decided by validation PR-AUC
final = [tune_and_evaluate(rf_tuned, "Random Forest Tuned"),
         tune_and_evaluate(cb_tuned, "CatBoost Tuned")]
final_table = pd.DataFrame(final).sort_values("Val PR-AUC", ascending=False).reset_index(drop=True)
print("\nWinner chosen by validation PR-AUC:", final_table.loc[0, "Model"])
final_table

{'Model': 'Random Forest Tuned', 'Threshold': np.float64(0.14), 'Val PR-AUC': np.float64(0.8837), 'PR-AUC': np.float64(0.8326), 'F1': 0.6887, 'F2': 0.7813, 'MCC': np.float64(0.7012)}
{'Model': 'CatBoost Tuned', 'Threshold': np.float64(0.09), 'Val PR-AUC': np.float64(0.882), 'PR-AUC': np.float64(0.8224), 'F1': 0.6331, 'F2': 0.7632, 'MCC': np.float64(0.6587)}

Winner chosen by validation PR-AUC: Random Forest Tuned


,Model,Threshold,Val PR-AUC,PR-AUC,F1,F2,MCC
0,Random Forest Tuned,0.14,0.8837,0.8326,0.6887,0.7813,0.7012
1,CatBoost Tuned,0.09,0.8820,0.8224,0.6331,0.7632,0.6587


**Effect of tuning on test:**

| Model | Test PR-AUC (default → tuned) | F1 | F2 | MCC |
|---|---|---|---|---|
| Random Forest | 0.8070 → **0.8326** | 0.6957 → 0.6887 | 0.7760 → 0.7813 | 0.7050 → 0.7012 |
| CatBoost | 0.8057 → 0.8224 | 0.6610 → 0.6331 | 0.7632 → 0.7632 | 0.6766 → 0.6587 |

- The validation winner, Random Forest Tuned, is also better than CatBoost Tuned on every test metric.
- Tuning carried over to test: Random Forest's test PR-AUC rose by 0.0256, more than its validation gain (+0.0174), so the improvement is not an artefact of fitting the validation set.
- Tuning mainly improved **ranking quality (PR-AUC)**. The threshold-based metrics barely moved (F2 up slightly; F1 and MCC down slightly), because the threshold is optimised for F2 on validation and that optimum does not transfer perfectly to the later test period.
- The validation-to-test gap narrowed slightly for Random Forest (0.059 → 0.051) but remains, consistent with the drift over time noted in the boosting stage.

## 8. Saving the selected model

The trained model and the category-risk map are saved, so new transactions can be scored later without retraining or re-tuning. Scoring takes seconds; only training is slow. The compressed model file is about 12 MB, and both files are stored in the repository's `results/` folder.

In [20]:
import joblib
from google.colab import files

os.makedirs('results', exist_ok=True)

# the selected model, plus the train-only category-risk map needed to rebuild its features
joblib.dump(rf_tuned, 'results/rf_tuned.joblib', compress=3)
joblib.dump(category_risk_map, 'results/category_risk_map.joblib')
print(f"model file: {os.path.getsize('results/rf_tuned.joblib')/1e6:.0f} MB")

# download local copies as a backup
files.download('results/rf_tuned.joblib')
files.download('results/category_risk_map.joblib')

model file: 12 MB


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## 9. Summary

We compared five models on the six-feature Sparkov set, all evaluated under the same protocol (80/20 train/validation split, F2-optimal threshold chosen on validation, one final test evaluation). At default settings, Random Forest led on test (PR-AUC 0.8070) with CatBoost a close second (0.8057). Weighting helped XGBoost but hurt CatBoost, and Balanced Random Forest came last (0.7207): building imbalance handling into the model did not beat a plain tree ensemble with a tuned threshold.

We then tuned Random Forest and CatBoost on validation PR-AUC only. **Random Forest Tuned** (200 trees, `max_features=3`, `min_samples_leaf=5`) won on validation (0.8837 vs 0.8820) and on test (**PR-AUC 0.8326, F1 0.6887, F2 0.7813, MCC 0.7012**). Tuning improved ranking quality (PR-AUC +0.026), while the threshold-based metrics changed little.

The gain from tuning (about 0.03 PR-AUC) is small next to the gain from model choice (about 0.46 PR-AUC from Logistic Regression to Random Forest). This reinforces the project's running finding that the choice of model matters more than the imbalance strategy or fine-tuning.

**Limitations:**
- Validation scores are slightly optimistic. The validation set was used for threshold tuning, hyperparameter selection and CatBoost early stopping, and `category_risk` was fit on the full training split, which includes the validation rows. Test scores are the figures we report.
- The shortlist of two models for tuning was drawn from the default-settings table, which includes test results. The final choice between them was made on validation only.
- All results come from a single random seed.
- A consistent validation-to-test drop of about 0.05 PR-AUC suggests drift over time in the Sparkov data.

**Selected model for Week 9 (IEEE-CIS validation, ablation and SHAP): Random Forest Tuned.** On IEEE-CIS, the threshold will be re-tuned on an IEEE-CIS validation split, and `max_features` will be checked against the number of features that transfer to that dataset.